# 07 · Evaluación, pipelines y MLOps: del notebook a producción

Un modelo no termina cuando obtiene una métrica alta. Necesitamos evaluación correcta, reproducibilidad, artefactos versionados, despliegue, observabilidad y monitoreo de drift/performance.

## Objetivos
- Elegir splits y métricas según el problema.
- Construir pipelines sin leakage.
- Usar cross-validation y búsqueda de hiperparámetros.
- Guardar un pipeline completo, no solo el estimador.
- Diseñar serving, monitoreo y rollback.
- Entender data drift, concept drift, training-serving skew y model decay.


In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, joblib, json, hashlib
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score, average_precision_score
SEED=42
X,y=load_breast_cancer(return_X_y=True,as_frame=True)
Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=.2,random_state=SEED,stratify=y)

## 1. La métrica debe representar el costo real
- Accuracy: útil con clases balanceadas y costos parecidos.
- Precision: minimiza falsos positivos.
- Recall: minimiza falsos negativos.
- F1: equilibrio armónico.
- ROC-AUC: ranking global, puede verse optimista con eventos muy raros.
- PR-AUC: muy informativa en clases raras.
- Log-loss/Brier: calidad probabilística/calibración.
- Regresión: MAE, RMSE, R², quantile loss, MAPE/SMAPE según contexto.

En sistemas reales también medimos latencia, throughput, costo, cobertura, fairness y métricas de negocio.


## 2. Pipeline = prevención de leakage + reproducibilidad
Toda transformación aprendida (imputación, scaling, encoding, feature selection) debe ajustarse **dentro** del fold de entrenamiento. `Pipeline` hace esto correctamente.


In [ ]:
pipe=Pipeline([
 ('impute',SimpleImputer(strategy='median')),
 ('scale',StandardScaler()),
 ('model',LogisticRegression(max_iter=5000,class_weight=None))
])
cv=StratifiedKFold(n_splits=5,shuffle=True,random_state=SEED)
scores=cross_validate(pipe,Xtr,ytr,cv=cv,scoring=['accuracy','precision','recall','f1','roc_auc','average_precision'],return_train_score=True,n_jobs=-1)
summary={k:(np.mean(v),np.std(v)) for k,v in scores.items() if k.startswith('test_')}
pd.DataFrame(summary,index=['mean','std']).T.round(4)

## 3. Hiperparámetros: búsqueda sin contaminar el test
El test final se toca una vez. `GridSearchCV`, `RandomizedSearchCV` u optimizadores bayesianos deben usar train/validation interno. Si hacemos muchas decisiones mirando test, el test se convierte de facto en validation.


In [ ]:
search=RandomizedSearchCV(pipe,{
 'model__C':np.logspace(-4,3,200),
 'model__penalty':['l2'],
 'model__class_weight':[None,'balanced']
},n_iter=30,scoring='average_precision',cv=cv,random_state=SEED,n_jobs=-1)
search.fit(Xtr,ytr); print(search.best_params_,search.best_score_)
best=search.best_estimator_; proba=best.predict_proba(Xte)[:,1]; pred=(proba>=.5).astype(int)
print('TEST ROC-AUC',roc_auc_score(yte,proba),'PR-AUC',average_precision_score(yte,proba)); print(classification_report(yte,pred,digits=3))

## 4. Nested CV y selección honesta
Cuando el dataset es pequeño y queremos estimar performance mientras tuneamos, nested CV usa un loop interno para tuning y otro externo para evaluación. Es más caro, pero evita el optimismo de reportar el mejor CV score usado para seleccionar el modelo.

También existen `GroupKFold`, `StratifiedGroupKFold` y `TimeSeriesSplit`; el split correcto depende de cómo aparecerán datos futuros.


## 5. Persistencia y trazabilidad
Guarda el pipeline completo junto con metadata: versión de código, schema, features, fecha, dataset/consulta, seed, hiperparámetros, métricas y dependencias. En sistemas maduros se usan model registries (MLflow, SageMaker, Vertex, Azure ML, Databricks).


In [ ]:
best.fit(Xtr,ytr); joblib.dump(best,'modelo_pipeline.joblib')
metadata={
 'model':'LogisticRegression','seed':SEED,'features':list(X.columns),'n_train':len(Xtr),
 'test_roc_auc':float(roc_auc_score(yte,best.predict_proba(Xte)[:,1]))
}
metadata['schema_sha256']=hashlib.sha256('|'.join(metadata['features']).encode()).hexdigest()
print(json.dumps(metadata,indent=2)[:1000])
loaded=joblib.load('modelo_pipeline.joblib'); print('reloaded score',loaded.score(Xte,yte))

## 6. Serving: batch, online y streaming
- **Batch:** predicciones periódicas sobre grandes tablas. Simple y barato.
- **Online API:** respuesta por solicitud; importa p95/p99 latency. FastAPI es una opción frecuente.
- **Streaming:** scoring continuo sobre eventos; requiere infraestructura y semántica de eventos.

En todos los casos valida input schema, tipos, rangos, features faltantes, versiones y manejo de errores.


## 7. Monitoreo: cuatro capas
1. **Sistema:** CPU/GPU, RAM, errores, latencia, throughput.
2. **Datos:** missing, rangos, schema, distribución, cardinalidad.
3. **Modelo:** score distributions, calibration, drift, incertidumbre.
4. **Negocio:** outcome real, costo/beneficio, cobertura, fairness y adopción.

Data drift $P(X)$ puede cambiar sin degradar modelo; concept drift $P(Y|X)$ es más peligroso y requiere etiquetas para detectarse directamente.


In [ ]:
# Ejemplo simple de Population Stability Index (PSI) para una feature
def psi(expected,actual,bins=10):
    cuts=np.quantile(expected,np.linspace(0,1,bins+1)); cuts[0],cuts[-1]=-np.inf,np.inf
    e=np.histogram(expected,bins=cuts)[0]/len(expected); a=np.histogram(actual,bins=cuts)[0]/len(actual)
    e=np.clip(e,1e-6,None); a=np.clip(a,1e-6,None)
    return np.sum((a-e)*np.log(a/e))
base=Xtr.iloc[:,0].to_numpy(); shifted=Xte.iloc[:,0].to_numpy()*1.25+2
print('PSI original test:',psi(base,Xte.iloc[:,0].to_numpy()))
print('PSI shifted:',psi(base,shifted))

## 8. Un ciclo MLOps razonable
`data/version → validation → train → evaluate → register → deploy → monitor → feedback → retrain/rollback`.

CI valida código/tests; CD despliega artefactos; CT (continuous training) reentrena cuando corresponde. Reentrenar siempre por calendario no es necesariamente mejor: puede introducir datos malos o degradar un modelo estable.

## Errores comunes
- serializar solo el estimador y olvidar preprocessing;
- no fijar schema/orden de features;
- no registrar versión de datos/código;
- monitorear solo uptime;
- hacer auto-retraining sin gates de calidad;
- no tener rollback/champion-challenger;
- permitir training-serving skew;
- medir fairness una sola vez y nunca en producción.

## Ejercicios
1. Implementa nested CV.
2. Empaqueta `modelo_pipeline.joblib` detrás de FastAPI `/predict`.
3. Crea tests con inputs válidos, missing y tipos incorrectos.
4. Simula drift de una feature y grafica PSI por semana.
5. Diseña un `model_card.md` con uso previsto, límites y métricas.
6. Registra el experimento en MLflow.
7. Diseña un flujo champion/challenger y criterios de rollback.
8. Compara batch scoring vs online API para un proceso de millones de registros.
